# Group C Full Run
This notebook runs the complete Group C Bark TTS workflow for HW13. It is intended for Colab GPU execution after the dry-run notebook has already been debugged successfully.

Expected outputs:
- 1 GA19C baseline audio file in `exp1_baselines`
- 20 Experiment 6 audio files in `exp6_ga19c_tts`
- 8 Experiment 7 seed-sensitivity audio files in `exp7_seed_sensitivity`
- 3 round-trip TTS to ASR transcriptions logged in the Experiment 6 CSV
- Full-run CSVs and console log output
- A timestamped `group_c_results_YYYYMMDD_HHMMSS.zip` bundle in `hw13_experiments`

If the Drive mirror is stale, this notebook will use notebook-local compatibility code for Group C while still writing the canonical HW13 outputs.

In [1]:
RUNNING_IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    RUNNING_IN_COLAB = True
    print('Running in Colab mode.')
except ImportError:
    print('google.colab not available; using local workspace mode.')

Mounted at /content/drive
Running in Colab mode.


In [2]:
%pip install -q transformers accelerate sentencepiece scipy soundfile librosa jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.6 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import os
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive')
LOCAL_ROOT = Path('/Users/noir/visual_studio/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13')
if RUNNING_IN_COLAB:
    CANDIDATE_ROOTS = [
        DRIVE_ROOT / 'kamp_hw13',
        DRIVE_ROOT / 'Visual_Studio__UC_Spring_26' / 'GEN_AI' / 'HW1' / 'kamp_hw13',
    ]
else:
    CANDIDATE_ROOTS = [
        LOCAL_ROOT,
        Path.cwd(),
        Path.cwd().parent,
    ]

def find_project_root():
    checked_paths = []
    for candidate in CANDIDATE_ROOTS:
        checked_paths.append(candidate)
        if (candidate / 'hw13_scripts' / 'kamp_hw13.py').exists():
            return candidate

    checked_display = '\n'.join(f' - {path}' for path in checked_paths)
    raise FileNotFoundError(
        'Could not locate kamp_hw13.\n'
        'In Colab, upload the project to /content/drive/MyDrive/kamp_hw13\n'
        'or mirror the local nesting under /content/drive/MyDrive/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13.\n'
        'In local mode, open the notebook from the kamp_hw13 workspace.\n'
        f'Checked:\n{checked_display}'
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'hw13_scripts'))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Module exists: {(PROJECT_ROOT / "hw13_scripts" / "kamp_hw13.py").exists()}')

PROJECT_ROOT = /content/drive/MyDrive/kamp_hw13
Module exists: True


In [4]:
import csv
import importlib
import json
from math import gcd

import numpy as np

try:
    import torch
except ImportError:
    torch = None

import hw13_data_utils
import hw13_experiment_runner
import kamp_hw13

importlib.reload(hw13_data_utils)
importlib.reload(hw13_experiment_runner)
importlib.reload(kamp_hw13)

from hw13_data_utils import (
    TeeLogger,
    append_csv_row,
    cleanup_pipeline,
    get_audio_stats,
    init_csv,
    now_iso,
    save_audio,
    timer,
    compute_wer_cer,
 )
from hw13_experiment_runner import run_ga19b

FULL_RUN_LOG = PROJECT_ROOT / 'hw13_printouts' / 'group_c_full_log.txt'

GROUP_A_EXP1_HEADER = getattr(kamp_hw13, 'GROUP_A_EXP1_HEADER', [
    'experiment', 'script', 'task_type', 'seed', 'prompt', 'ip_adapter_scale',
    'num_inference_steps', 'model', 'reference_image', 'input_text', 'output_path',
    'generation_time_sec', 'timestamp',
])
GROUP_A_EXP7_HEADER = getattr(kamp_hw13, 'GROUP_A_EXP7_HEADER', [
    'experiment', 'script', 'paradigm', 'seed', 'prompt', 'parameters_json',
    'output_path', 'output_type', 'generation_time_sec', 'timestamp',
])
SEEDS_2 = getattr(kamp_hw13, 'SEEDS_2', [42, 123])
SEEDS_8 = getattr(kamp_hw13, 'SEEDS_8', [42, 123, 456, 789, 1024, 2048, 3000, 4096])

GROUP_C_TTS_MODEL = getattr(kamp_hw13, 'GROUP_C_TTS_MODEL', 'suno/bark-small')
GROUP_C_ROUND_TRIP_FALLBACK_ASR_MODEL = getattr(
    kamp_hw13,
    'GROUP_C_ROUND_TRIP_FALLBACK_ASR_MODEL',
    'openai/whisper-medium',
)
GA19C_DEFAULT_TEXT = getattr(
    kamp_hw13,
    'GA19C_DEFAULT_TEXT',
    'Ladybugs have had important roles in culture and religion, being associated with luck, love, fertility and prophecy.',
)
GROUP_C_EXP6A_SEEDS = getattr(kamp_hw13, 'GROUP_C_EXP6A_SEEDS', [42, 123, 456, 789, 1024])
GROUP_C_EXP6B_TEXTS = getattr(kamp_hw13, 'GROUP_C_EXP6B_TEXTS', [
    {'text_tag': 'short', 'text_category': 'short', 'input_text': 'Hello, how are you?'},
    {'text_tag': 'medium', 'text_category': 'medium', 'input_text': GA19C_DEFAULT_TEXT},
    {
        'text_tag': 'long',
        'text_category': 'long',
        'input_text': 'The quick brown fox jumps over the lazy dog. This sentence contains every letter in the English alphabet and has been used for typing practice since the late nineteenth century.',
    },
])
GROUP_C_EXP6C_TEXTS = getattr(kamp_hw13, 'GROUP_C_EXP6C_TEXTS', [
    {
        'text_tag': 'conversational',
        'text_category': 'conversational',
        'input_text': 'Hey, could you grab me a coffee on your way back? Thanks a lot!',
    },
    {
        'text_tag': 'technical',
        'text_category': 'technical',
        'input_text': 'The convolutional neural network processes input tensors through sequential layers of learned filters.',
    },
    {
        'text_tag': 'proper_nouns',
        'text_category': 'proper_nouns',
        'input_text': 'Professor Schwarzenegger from the University of Cincinnati presented the findings at the symposium.',
    },
])
GROUP_C_EXP6D_TEXTS = getattr(kamp_hw13, 'GROUP_C_EXP6D_TEXTS', [
    {
        'text_tag': 'roundtrip_short',
        'text_category': 'short',
        'input_text': GROUP_C_EXP6B_TEXTS[0]['input_text'],
    },
    {
        'text_tag': 'roundtrip_technical',
        'text_category': 'technical',
        'input_text': GROUP_C_EXP6C_TEXTS[1]['input_text'],
    },
    {
        'text_tag': 'roundtrip_proper_nouns',
        'text_category': 'proper_nouns',
        'input_text': GROUP_C_EXP6C_TEXTS[2]['input_text'],
    },
])
GROUP_C_EXP6_HEADER = getattr(kamp_hw13, 'GROUP_C_EXP6_HEADER', [
    'experiment', 'sub_experiment', 'script', 'seed', 'model', 'input_text',
    'text_length_words', 'text_category', 'output_path', 'audio_duration_sec',
    'audio_sampling_rate', 'waveform_rms', 'generation_time_sec',
    'round_trip_transcription', 'round_trip_wer', 'timestamp',
])

def _relative_output_path(path, experiments_root):
    path = Path(path)
    experiments_root = Path(experiments_root)
    try:
        return str(path.relative_to(experiments_root))
    except ValueError:
        return str(path.name)

def _prepare_group_c_layout(experiments_root):
    experiments_root = Path(experiments_root)
    return {
        'root': experiments_root,
        'exp1_dir': experiments_root / 'exp1_baselines',
        'exp6_dir': experiments_root / 'exp6_ga19c_tts',
        'exp7_dir': experiments_root / 'exp7_seed_sensitivity',
    }

def _select_best_group_b_asr_model(
    exp5_csv_path=None,
    comparison_sub_experiment='5a',
    fallback_model=GROUP_C_ROUND_TRIP_FALLBACK_ASR_MODEL,
 ):
    csv_path = Path(exp5_csv_path or (PROJECT_ROOT / 'hw13_experiments' / 'exp5_ga19b_asr' / 'exp5_ga19b_asr.csv'))
    if not csv_path.exists():
        print(f'[NOTEBOOK] Experiment 5 CSV missing at {csv_path}; using fallback ASR model {fallback_model}.')
        return fallback_model

    aggregates = {}
    with csv_path.open('r', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            if comparison_sub_experiment and row.get('sub_experiment') != comparison_sub_experiment:
                continue
            model_name = (row.get('model_name') or '').strip()
            wer_text = row.get('wer', '')
            cer_text = row.get('cer', '')
            if not model_name or not wer_text or not cer_text:
                continue
            stats = aggregates.setdefault(model_name, {'count': 0, 'wer_sum': 0.0, 'cer_sum': 0.0})
            stats['count'] += 1
            stats['wer_sum'] += float(wer_text)
            stats['cer_sum'] += float(cer_text)

    if not aggregates:
        print(f'[NOTEBOOK] No usable Experiment 5 rows found; using fallback ASR model {fallback_model}.')
        return fallback_model

    ranked_models = sorted(
        (stats['wer_sum'] / stats['count'], stats['cer_sum'] / stats['count'], -stats['count'], model_name)
        for model_name, stats in aggregates.items()
    )
    best_wer, best_cer, _, best_model = ranked_models[0]
    print(
        f'[NOTEBOOK] Selected round-trip ASR model: {best_model} '
        f'(avg WER={best_wer:.4f}, avg CER={best_cer:.4f}, subset={comparison_sub_experiment})'
    )
    return best_model

def _resample_audio_for_asr(audio_array, source_rate, target_rate=16000):
    audio_np = np.asarray(audio_array, dtype=np.float32)
    audio_np = np.squeeze(audio_np)
    if not source_rate or int(source_rate) == int(target_rate):
        return audio_np

    from scipy.signal import resample_poly

    divisor = gcd(int(source_rate), int(target_rate))
    up = int(target_rate) // divisor
    down = int(source_rate) // divisor
    return resample_poly(audio_np, up, down).astype(np.float32)

def load_group_c_tts_pipeline(device=None):
    from transformers import pipeline

    resolved_device = device if device is not None else (0 if torch is not None and torch.cuda.is_available() else -1)
    tts_pipeline = pipeline('text-to-speech', model=GROUP_C_TTS_MODEL, device=resolved_device)
    return tts_pipeline, GROUP_C_TTS_MODEL, resolved_device

def run_ga19c_local(
    tts_pipeline,
    input_text,
    seed,
    save_dir,
    csv_path=None,
    sub_experiment='',
    extra_meta=None,
 ):
    metadata = dict(extra_meta or {})
    save_dir = Path(save_dir)
    csv_path = Path(csv_path) if csv_path else None

    if torch is None:
        raise ImportError('torch is required for Group C notebook execution.')

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    with timer(f"GA19C text='{input_text[:40]}...' seed={seed}") as timing:
        output = tts_pipeline(input_text)

    audio_array = output['audio']
    sampling_rate = output['sampling_rate']
    stats = get_audio_stats(audio_array)
    duration_sec = stats['duration_samples'] / sampling_rate if sampling_rate else 0.0

    text_tag = metadata.get('text_tag', 'default')
    audio_name = f'ga19c_{sub_experiment}_{text_tag}_seed{seed}.wav'
    audio_path = save_audio(audio_array, sampling_rate, save_dir / audio_name, label=audio_name)

    timestamp = now_iso()
    row = [
        metadata.get('experiment', 'exp6'),
        sub_experiment,
        'GA19C',
        seed,
        metadata.get('model_name', GROUP_C_TTS_MODEL),
        input_text,
        len(input_text.split()),
        metadata.get('text_category', 'default'),
        str(audio_path.name),
        round(duration_sec, 3),
        sampling_rate,
        round(stats['rms'], 6),
        timing['elapsed_sec'],
        metadata.get('round_trip_transcription', ''),
        metadata.get('round_trip_wer', ''),
        timestamp,
    ]
    if csv_path is not None:
        append_csv_row(csv_path, row)
    return {
        'audio': audio_array,
        'sampling_rate': sampling_rate,
        'duration': duration_sec,
        'stats': stats,
        'time': timing['elapsed_sec'],
        'path': audio_path,
        'timestamp': timestamp,
    }

def run_group_c_local(
    dry_run=False,
    experiments_dir=None,
    checkpoint_enabled=True,
    resume=False,
 ):
    layout = _prepare_group_c_layout(experiments_dir or (PROJECT_ROOT / 'hw13_experiments'))
    experiments_root = layout['root']

    print(f"\n{'=' * 60}")
    print(f"[GROUP C] Starting {'dry-run' if dry_run else 'full'} Bark TTS workflow")
    print(f"[GROUP C] Output root: {experiments_root}")

    exp1_csv = init_csv(layout['exp1_dir'] / 'exp1_baselines.csv', GROUP_A_EXP1_HEADER)
    exp6_csv = init_csv(layout['exp6_dir'] / 'exp6_ga19c_tts.csv', GROUP_C_EXP6_HEADER)
    exp7_csv = init_csv(layout['exp7_dir'] / 'exp7_seed_sensitivity.csv', GROUP_A_EXP7_HEADER)

    summary = {
        'mode': 'dry-run' if dry_run else 'full',
        'output_root': str(experiments_root),
        'tts_model': GROUP_C_TTS_MODEL,
        'best_asr_model': '',
        'round_trip_asr_model': '',
        'total_audio_files': 0,
        'round_trip_transcriptions': 0,
        'per_experiment': {'exp1': 0, 'exp6': 0, 'exp7': 0},
    }

    def record_audio(experiment_key):
        summary['total_audio_files'] += 1
        summary['per_experiment'][experiment_key] += 1

    best_asr_model = _select_best_group_b_asr_model()
    summary['best_asr_model'] = best_asr_model

    baseline_seed = 42
    exp6a_seeds = [baseline_seed] if dry_run else GROUP_C_EXP6A_SEEDS
    exp6b_texts = [GROUP_C_EXP6B_TEXTS[0]] if dry_run else GROUP_C_EXP6B_TEXTS
    exp6c_texts = [GROUP_C_EXP6C_TEXTS[1]] if dry_run else GROUP_C_EXP6C_TEXTS
    exp6d_texts = [GROUP_C_EXP6D_TEXTS[1]] if dry_run else GROUP_C_EXP6D_TEXTS
    exp6_variant_seeds = [baseline_seed] if dry_run else SEEDS_2
    exp7_seeds = [baseline_seed] if dry_run else SEEDS_8

    tts_pipeline = None
    resolved_tts_model = GROUP_C_TTS_MODEL
    try:
        tts_pipeline, resolved_tts_model, resolved_device = load_group_c_tts_pipeline()
        summary['tts_model'] = resolved_tts_model
        print(f'[GROUP C] Loaded Bark pipeline: {resolved_tts_model} on device {resolved_device}')

        baseline_result = run_ga19c_local(
            tts_pipeline,
            input_text=GA19C_DEFAULT_TEXT,
            seed=baseline_seed,
            save_dir=layout['exp1_dir'],
            csv_path=None,
            sub_experiment='baseline',
            extra_meta={
                'experiment': 'exp1',
                'model_name': resolved_tts_model,
                'text_tag': 'default',
                'text_category': 'default',
            },
        )
        append_csv_row(
            exp1_csv,
            [
                'exp1',
                'GA19C',
                'text_to_speech',
                baseline_seed,
                '',
                '',
                '',
                resolved_tts_model,
                '',
                GA19C_DEFAULT_TEXT,
                _relative_output_path(baseline_result['path'], experiments_root),
                baseline_result['time'],
                baseline_result['timestamp'],
            ],
        )
        record_audio('exp1')

        for seed in exp6a_seeds:
            run_ga19c_local(
                tts_pipeline,
                input_text=GA19C_DEFAULT_TEXT,
                seed=seed,
                save_dir=layout['exp6_dir'],
                csv_path=exp6_csv,
                sub_experiment='6a',
                extra_meta={
                    'experiment': 'exp6',
                    'model_name': resolved_tts_model,
                    'text_tag': 'repeat',
                    'text_category': 'repeat',
                },
            )
            record_audio('exp6')

        for config in exp6b_texts:
            for seed in exp6_variant_seeds:
                run_ga19c_local(
                    tts_pipeline,
                    input_text=config['input_text'],
                    seed=seed,
                    save_dir=layout['exp6_dir'],
                    csv_path=exp6_csv,
                    sub_experiment='6b',
                    extra_meta={
                        'experiment': 'exp6',
                        'model_name': resolved_tts_model,
                        'text_tag': config['text_tag'],
                        'text_category': config['text_category'],
                    },
                )
                record_audio('exp6')

        for config in exp6c_texts:
            for seed in exp6_variant_seeds:
                run_ga19c_local(
                    tts_pipeline,
                    input_text=config['input_text'],
                    seed=seed,
                    save_dir=layout['exp6_dir'],
                    csv_path=exp6_csv,
                    sub_experiment='6c',
                    extra_meta={
                        'experiment': 'exp6',
                        'model_name': resolved_tts_model,
                        'text_tag': config['text_tag'],
                        'text_category': config['text_category'],
                    },
                )
                record_audio('exp6')

        asr_pipeline = None
        resolved_round_trip_model = best_asr_model
        try:
            asr_pipeline, resolved_round_trip_model, resolved_asr_device = kamp_hw13.load_group_b_asr_pipeline(best_asr_model)
            summary['round_trip_asr_model'] = resolved_round_trip_model
            print(f'[GROUP C] Loaded round-trip ASR pipeline: {resolved_round_trip_model} on device {resolved_asr_device}')
            for config in exp6d_texts:
                tts_result = run_ga19c_local(
                    tts_pipeline,
                    input_text=config['input_text'],
                    seed=baseline_seed,
                    save_dir=layout['exp6_dir'],
                    csv_path=None,
                    sub_experiment='6d',
                    extra_meta={
                        'experiment': 'exp6',
                        'model_name': resolved_tts_model,
                        'text_tag': config['text_tag'],
                        'text_category': config['text_category'],
                    },
                )
                asr_audio = _resample_audio_for_asr(tts_result['audio'], tts_result['sampling_rate'], target_rate=16000)
                asr_result = run_ga19b(
                    asr_pipeline,
                    audio_array=asr_audio,
                    sampling_rate=16000,
                    ground_truth=config['input_text'],
                    model_name=resolved_round_trip_model,
                    sample_index=-1,
                    csv_path=None,
                    sub_experiment='6d',
                    extra_meta={'experiment': 'exp6'},
                )
                append_csv_row(
                    exp6_csv,
                    [
                        'exp6',
                        '6d',
                        'GA19C',
                        baseline_seed,
                        resolved_tts_model,
                        config['input_text'],
                        len(config['input_text'].split()),
                        config['text_category'],
                        str(tts_result['path'].name),
                        round(tts_result['duration'], 3),
                        tts_result['sampling_rate'],
                        round(tts_result['stats']['rms'], 6),
                        tts_result['time'],
                        asr_result['predicted'],
                        asr_result['wer'],
                        tts_result['timestamp'],
                    ],
                )
                record_audio('exp6')
                summary['round_trip_transcriptions'] += 1
        finally:
            if asr_pipeline is not None:
                cleanup_pipeline(asr_pipeline, f'ASR round-trip {resolved_round_trip_model}')

        for seed in exp7_seeds:
            result = run_ga19c_local(
                tts_pipeline,
                input_text=GA19C_DEFAULT_TEXT,
                seed=seed,
                save_dir=layout['exp7_dir'],
                csv_path=None,
                sub_experiment='seed',
                extra_meta={
                    'experiment': 'exp7',
                    'model_name': resolved_tts_model,
                    'text_tag': 'default',
                    'text_category': 'seed_sensitivity',
                },
            )
            append_csv_row(
                exp7_csv,
                [
                    'exp7',
                    'GA19C',
                    'text_to_speech',
                    seed,
                    GA19C_DEFAULT_TEXT,
                    json.dumps({
                        'model': resolved_tts_model,
                        'text_category': 'seed_sensitivity',
                        'sub_experiment': 'seed',
                    }, sort_keys=True),
                    _relative_output_path(result['path'], experiments_root),
                    'audio',
                    result['time'],
                    result['timestamp'],
                ],
            )
            record_audio('exp7')

        print(
            f"[GROUP C] Complete. Audio files={summary['total_audio_files']}, "
            f"round-trip transcriptions={summary['round_trip_transcriptions']} ({summary['per_experiment']})."
        )
        return summary
    finally:
        if tts_pipeline is not None:
            cleanup_pipeline(tts_pipeline, f'Bark TTS {resolved_tts_model}')

if hasattr(kamp_hw13, '_select_best_group_b_asr_model'):
    BEST_ASR_MODEL = kamp_hw13._select_best_group_b_asr_model()
    run_group_c = kamp_hw13.run_group_c
    print('[NOTEBOOK] Using shared Group C implementation from kamp_hw13.py')
else:
    BEST_ASR_MODEL = _select_best_group_b_asr_model()
    run_group_c = run_group_c_local
    print('[NOTEBOOK] Drive copy is stale; using notebook-local Group C compatibility implementation.')

print(f'Using Group C implementation from {kamp_hw13.__file__}')
print(f'BEST_ASR_MODEL = {BEST_ASR_MODEL}')
print(f'FULL_RUN_LOG = {FULL_RUN_LOG}')

[NOTEBOOK] Selected round-trip ASR model: openai/whisper-medium (avg WER=0.1715, avg CER=0.1000, subset=5a)
[NOTEBOOK] Drive copy is stale; using notebook-local Group C compatibility implementation.
Using Group C implementation from /content/drive/MyDrive/kamp_hw13/hw13_scripts/kamp_hw13.py
BEST_ASR_MODEL = openai/whisper-medium
FULL_RUN_LOG = /content/drive/MyDrive/kamp_hw13/hw13_printouts/group_c_full_log.txt


In [5]:
if not hasattr(TeeLogger, 'isatty'):
    def _tee_isatty(self):
        stdout_isatty = getattr(self.stdout, 'isatty', None)
        if callable(stdout_isatty):
            return bool(stdout_isatty())
        return False

    def _tee_fileno(self):
        stdout_fileno = getattr(self.stdout, 'fileno', None)
        if callable(stdout_fileno):
            return stdout_fileno()
        raise OSError('Underlying stdout does not expose fileno()')

    def _tee_getattr(self, name):
        return getattr(self.stdout, name)

    TeeLogger.isatty = _tee_isatty
    TeeLogger.fileno = _tee_fileno
    TeeLogger.encoding = property(lambda self: getattr(self.stdout, 'encoding', 'utf-8'))
    TeeLogger.__getattr__ = _tee_getattr
    print('[NOTEBOOK] Patched TeeLogger for Colab stream compatibility.')
else:
    print('[NOTEBOOK] TeeLogger already exposes stream-compatibility helpers.')

[NOTEBOOK] Patched TeeLogger for Colab stream compatibility.


In [6]:
if not hasattr(kamp_hw13, 'load_group_b_asr_pipeline'):
    def _resolve_asr_model_name_local(asr_pipeline, fallback_name):
        if isinstance(asr_pipeline, dict):
            return asr_pipeline.get('model_name') or fallback_name or 'default_pipeline'
        model = getattr(asr_pipeline, 'model', None)
        config = getattr(model, 'config', None)
        resolved_name = getattr(config, '_name_or_path', None)
        return resolved_name or fallback_name or 'default_pipeline'

    def load_group_b_asr_pipeline_local(model_name=None, device=None):
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

        resolved_device = device if device is not None else (0 if torch is not None and torch.cuda.is_available() else -1)

        if model_name is not None and 'whisper' in model_name.lower():
            processor = AutoProcessor.from_pretrained(model_name)
            model_kwargs = {}
            if torch is not None:
                if resolved_device != -1 and torch.cuda.is_available():
                    model_kwargs['dtype'] = torch.float16
                else:
                    model_kwargs['dtype'] = torch.float32
            model = AutoModelForSpeechSeq2Seq.from_pretrained(model_name, **model_kwargs)
            if torch is not None and resolved_device != -1 and torch.cuda.is_available():
                model = model.to('cuda')
            return ({
                'backend_type': 'whisper_generate',
                'model_name': model_name,
                'processor': processor,
                'model': model,
            }, model_name, resolved_device)

        pipeline_kwargs = {'device': resolved_device}
        if model_name is not None:
            pipeline_kwargs['model'] = model_name

        asr_pipeline = pipeline('automatic-speech-recognition', **pipeline_kwargs)
        resolved_name = _resolve_asr_model_name_local(asr_pipeline, model_name)
        return asr_pipeline, resolved_name, resolved_device

    kamp_hw13.load_group_b_asr_pipeline = load_group_b_asr_pipeline_local
    print('[NOTEBOOK] Added notebook-local Group B ASR loader compatibility.')
else:
    print('[NOTEBOOK] Shared Group B ASR loader already available.')

[NOTEBOOK] Added notebook-local Group B ASR loader compatibility.


In [7]:
def run_ga19b_local(
    asr_pipeline,
    audio_array,
    sampling_rate,
    ground_truth,
    model_name,
    sample_index,
    csv_path=None,
    sub_experiment='',
    extra_meta=None,
 ):
    metadata = dict(extra_meta or {})
    csv_path = Path(csv_path) if csv_path else None

    with timer(f'GA19B model={model_name} sample={sample_index}') as timing:
        if isinstance(asr_pipeline, dict) and asr_pipeline.get('backend_type') == 'whisper_generate':
            if torch is None:
                raise ImportError('torch is required for Whisper round-trip generation.')
            processor = asr_pipeline['processor']
            model = asr_pipeline['model']
            audio_np = np.asarray(audio_array, dtype=np.float32)
            effective_sampling_rate = 16000 if sub_experiment == '5b' else (sampling_rate or 16000)
            model_inputs = processor(
                audio_np,
                sampling_rate=effective_sampling_rate,
                return_attention_mask=True,
                return_tensors='pt',
            )

            input_features = model_inputs['input_features'].to(model.device)
            model_dtype = getattr(model, 'dtype', None)
            if model_dtype is not None:
                input_features = input_features.to(model_dtype)

            generate_kwargs = {'language': 'en', 'task': 'transcribe'}
            attention_mask = model_inputs.get('attention_mask')
            if attention_mask is not None:
                generate_kwargs['attention_mask'] = attention_mask.to(model.device)

            with torch.no_grad():
                predicted_ids = model.generate(input_features, **generate_kwargs)
            predicted = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        else:
            result = asr_pipeline(audio_array)
            predicted = result['text']

    wer_value, cer_value = compute_wer_cer(ground_truth, predicted)
    audio_duration = len(audio_array) / sampling_rate if sampling_rate else 0.0
    timestamp = now_iso()

    if csv_path is not None:
        row = [
            metadata.get('experiment', 'exp5'),
            sub_experiment,
            'GA19B',
            model_name,
            sample_index,
            round(audio_duration, 3),
            sampling_rate,
            ground_truth,
            predicted,
            wer_value,
            cer_value,
            timing['elapsed_sec'],
            timestamp,
        ]
        append_csv_row(csv_path, row)

    return {
        'predicted': predicted,
        'wer': wer_value,
        'cer': cer_value,
        'time': timing['elapsed_sec'],
        'timestamp': timestamp,
    }

run_ga19b = run_ga19b_local
print('[NOTEBOOK] Added notebook-local GA19B runner compatibility.')

[NOTEBOOK] Added notebook-local GA19B runner compatibility.


In [8]:
tee = TeeLogger(FULL_RUN_LOG)
tee_started = False

try:
    tee.start()
    tee_started = True
    print('[NOTEBOOK] Starting Group C full run')
    summary = run_group_c(
        dry_run=False,
        experiments_dir=PROJECT_ROOT / 'hw13_experiments',
        checkpoint_enabled=True,
        resume=True,
    )
    print(summary)
finally:
    if tee_started:
        tee.stop()

[NOTEBOOK] Starting Group C full run

[GROUP C] Starting full Bark TTS workflow
[GROUP C] Output root: /content/drive/MyDrive/kamp_hw13/hw13_experiments
[CSV] Initialized: /content/drive/MyDrive/kamp_hw13/hw13_experiments/exp6_ga19c_tts/exp6_ga19c_tts.csv
[NOTEBOOK] Selected round-trip ASR model: openai/whisper-medium (avg WER=0.1715, avg CER=0.1000, subset=5a)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_layers.1.weight to fine_acoustics.lm_heads.0.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_layers.2.weight to fine_acoustics.lm_heads.1.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_layers.3.weight to fine_acoustics.lm_heads.2.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_l

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

[GROUP C] Loaded round-trip ASR pipeline: openai/whisper-medium on device 0
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:/

In [9]:
from datetime import datetime
import shutil

bundle_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
bundle_root = PROJECT_ROOT / 'hw13_experiments' / f'group_c_results_{bundle_timestamp}'
bundle_logs_dir = bundle_root / 'logs'

bundle_root.mkdir(parents=True, exist_ok=True)
bundle_logs_dir.mkdir(parents=True, exist_ok=True)

copy_specs = [
    (PROJECT_ROOT / 'hw13_experiments' / 'exp1_baselines', bundle_root / 'exp1_baselines', ['exp1_baselines.csv', 'ga19c_*.wav']),
    (PROJECT_ROOT / 'hw13_experiments' / 'exp6_ga19c_tts', bundle_root / 'exp6_ga19c_tts', ['exp6_ga19c_tts.csv', 'ga19c_*.wav']),
    (PROJECT_ROOT / 'hw13_experiments' / 'exp7_seed_sensitivity', bundle_root / 'exp7_seed_sensitivity', ['exp7_seed_sensitivity.csv', 'ga19c_*.wav']),
]

for src_dir, dst_dir, patterns in copy_specs:
    dst_dir.mkdir(parents=True, exist_ok=True)
    for pattern in patterns:
        for path in src_dir.glob(pattern):
            shutil.copy2(path, dst_dir / path.name)
            print(f'Copied: {path.name}')

if FULL_RUN_LOG.exists():
    shutil.copy2(FULL_RUN_LOG, bundle_logs_dir / FULL_RUN_LOG.name)
    print(f'Copied log: {FULL_RUN_LOG.name}')

zip_path = shutil.make_archive(str(bundle_root), 'zip', root_dir=bundle_root.parent, base_dir=bundle_root.name)
print(f'Bundle directory: {bundle_root}')
print(f'ZIP archive: {zip_path}')

Copied: exp1_baselines.csv
Copied: ga19c_baseline_default_seed42.wav
Copied: exp6_ga19c_tts.csv
Copied: ga19c_6a_repeat_seed42.wav
Copied: ga19c_6a_repeat_seed123.wav
Copied: ga19c_6a_repeat_seed456.wav
Copied: ga19c_6a_repeat_seed789.wav
Copied: ga19c_6a_repeat_seed1024.wav
Copied: ga19c_6b_short_seed42.wav
Copied: ga19c_6b_short_seed123.wav
Copied: ga19c_6b_medium_seed42.wav
Copied: ga19c_6b_medium_seed123.wav
Copied: ga19c_6b_long_seed42.wav
Copied: ga19c_6b_long_seed123.wav
Copied: ga19c_6c_conversational_seed42.wav
Copied: ga19c_6c_conversational_seed123.wav
Copied: ga19c_6c_technical_seed42.wav
Copied: ga19c_6c_technical_seed123.wav
Copied: ga19c_6c_proper_nouns_seed42.wav
Copied: ga19c_6c_proper_nouns_seed123.wav
Copied: ga19c_6d_roundtrip_short_seed42.wav
Copied: ga19c_6d_roundtrip_technical_seed42.wav
Copied: ga19c_6d_roundtrip_proper_nouns_seed42.wav
Copied: exp7_seed_sensitivity.csv
Copied: ga19c_seed_default_seed42.wav
Copied: ga19c_seed_default_seed123.wav
Copied: ga19c_se

In [10]:
import csv

exp1_audio = sorted((PROJECT_ROOT / 'hw13_experiments' / 'exp1_baselines').glob('ga19c_*.wav'))
exp6_audio = sorted((PROJECT_ROOT / 'hw13_experiments' / 'exp6_ga19c_tts').glob('ga19c_*.wav'))
exp7_audio = sorted((PROJECT_ROOT / 'hw13_experiments' / 'exp7_seed_sensitivity').glob('ga19c_*.wav'))
exp6_csv = PROJECT_ROOT / 'hw13_experiments' / 'exp6_ga19c_tts' / 'exp6_ga19c_tts.csv'

assert len(exp1_audio) >= 1, f'Expected at least 1 GA19C baseline audio file, found {len(exp1_audio)}'
assert len(exp6_audio) >= 16, f'Expected at least 16 Exp 6 audio files, found {len(exp6_audio)}'
assert len(exp7_audio) >= 8, f'Expected at least 8 Exp 7 audio files, found {len(exp7_audio)}'
assert exp6_csv.exists(), f'Missing Exp 6 CSV: {exp6_csv}'

with exp6_csv.open() as handle:
    exp6_rows = list(csv.DictReader(handle))

round_trip_rows = [row for row in exp6_rows if row.get('round_trip_transcription')]
print(f'Baseline audio files: {len(exp1_audio)}')
print(f'Exp 6 audio files: {len(exp6_audio)}')
print(f'Exp 7 audio files: {len(exp7_audio)}')
print(f'Exp 6 rows: {len(exp6_rows)}')
print(f'Round-trip rows: {len(round_trip_rows)}')

assert len(exp6_rows) >= 16, f'Expected at least 16 Exp 6 rows, found {len(exp6_rows)}'
assert len(round_trip_rows) >= 2, f'Expected at least 2 round-trip rows, found {len(round_trip_rows)}'
print('Group C full-run post-checks completed successfully.')

Baseline audio files: 1
Exp 6 audio files: 20
Exp 7 audio files: 8
Exp 6 rows: 20
Round-trip rows: 3
Group C full-run post-checks completed successfully.
